In [ ]:
# CSV'nin ilk 5 satırını görelim
with open(TRAIN_CSV, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        print(f"Satır {i}: {line[:200]}")  # İlk 200 karakter
        if i >= 4: break

Satır 0: 6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,
Satır 1: 5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,
Satır 2: 7,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,
Satır 3: 9,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,
Satır 4: 5,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0

In [ ]:
import os

print("--- DOSYA KONTROLÜ ---")
print(f"Eğitim (Train) Dosyası Boyutu: {os.path.getsize(TRAIN_CSV)} byte")
print(f"Doğrulama (Val) Dosyası Boyutu: {os.path.getsize(VAL_CSV)} byte")

if os.path.getsize(TRAIN_CSV) == os.path.getsize(VAL_CSV):
    print("\n🚨 KIRMIZI ALARM: İki dosya tıpatıp AYNI! Model kopya çekiyor.")
    print("👉 Çözüm: Val RAR'ının içine yanlış dosya koymuş olabilirsin, kontrol etmelisin.")
else:
    print("\n✅ Dosyalar farklı. Kopya çekme yok, sorun benim eklediğim matematiksel filtrede olabilir.")

--- DOSYA KONTROLÜ ---
Eğitim (Train) Dosyası Boyutu: 36523880 byte
Doğrulama (Val) Dosyası Boyutu: 18289443 byte

✅ Dosyalar farklı. Kopya çekme yok, sorun benim eklediğim matematiksel filtrede olabilir.


In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
from google.colab import drive

# ----------------- 1. BAĞLANTI -----------------
print("🔄 Drive'a bağlanılıyor...")
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

# CSV'leri direkt belirt (RAR yok artık)
TRAIN_CSV = '/content/drive/MyDrive/video_koordinat_verisetiTRAİN.csv'
VAL_CSV   = '/content/drive/MyDrive/video_koordinat_verisetival2.csv'

# Dosyaların gerçekten var olduğunu kontrol et
assert os.path.exists(TRAIN_CSV), f"❌ Train CSV bulunamadı: {TRAIN_CSV}"
assert os.path.exists(VAL_CSV),   f"❌ Val CSV bulunamadı: {VAL_CSV}"
print(f"✅ Train: {TRAIN_CSV}")
print(f"✅ Val:   {VAL_CSV}")

# ----------------- 2. DOĞRULAMA -----------------
print("\n🔍 CSV yapısı kontrol ediliyor...")
with open(TRAIN_CSV, 'r', encoding='utf-8') as f:
    header_line = f.readline().strip()
    first_line  = f.readline().strip()

print(f"Başlık (ilk 80 kar): {header_line[:80]}")
print(f"İlk satır (ilk 80 kar): {first_line[:80]}")

# Başlığın doğru formatta olduğunu kontrol et
assert 'video_name' in header_line, "❌ Başlıkta 'video_name' yok — CSV formatı yanlış!"
assert 'class_id'   in header_line, "❌ Başlıkta 'class_id' yok!"
print("✅ CSV formatı doğru")

# ----------------- 3. GERÇEK VİDEO SAYISINI BUL -----------------
def count_videos(filepath):
    count = 0
    prev  = None
    with open(filepath, 'r', encoding='utf-8') as f:
        f.readline()  # header atla
        for line in f:
            parts = line.strip().split(',')
            if len(parts) > 0 and parts[0] != prev:
                count += 1
                prev = parts[0]
    return count

print("\n📊 Video sayıları hesaplanıyor (biraz sürebilir)...")
train_count = count_videos(TRAIN_CSV)
val_count   = count_videos(VAL_CSV)
print(f"Train: {train_count} video")
print(f"Val:   {val_count} video")

# ----------------- 4. HESAPLAMA AYARLARI -----------------
with open(TRAIN_CSV, 'r', encoding='utf-8') as f:
    header = f.readline().strip().split(',')

# header: [video_name, class_id, frame_no, feat1, feat2, ...]
feature_names = header[3:]

# Bacakları ve alt vücudu filtrele (pose 25+ gürültü)
bad_indices = set()
for i, col_name in enumerate(feature_names):
    if 'pose_' in col_name:
        try:
            p_num = int(col_name.split('_')[1])
            if p_num >= 25:
                bad_indices.add(i)
        except: pass

valid_indices = [i for i in range(len(feature_names)) if i not in bad_indices]

def get_idx(name):
    try: return feature_names.index(name)
    except: return -1

idx_nose_x = get_idx('pose_0_x')
idx_nose_y = get_idx('pose_0_y')

ADDED_FEATURES  = 0          # Şimdilik mesafe yok, sıfır kafa karıştırmasın
NEW_NUM_FEATURES = len(valid_indices) + ADDED_FEATURES
NUM_CLASSES = 226
MAX_FRAMES  = 100
BATCH_SIZE  = 32

# Gerçek adım sayıları (sabit değil, hesaplanmış)
TRAIN_STEPS = train_count // BATCH_SIZE
VAL_STEPS   = val_count   // BATCH_SIZE

print(f"\n📐 Özellik sayısı: {NEW_NUM_FEATURES}")
print(f"📐 Train adım: {TRAIN_STEPS} | Val adım: {VAL_STEPS}")

# ----------------- 5. GENERATOR -----------------
def robust_generator(filepath):
    while True:
        with open(filepath, 'r', encoding='utf-8', errors='replace') as f:
            f.readline()  # header atla
            current_video, current_label, frames = None, None, []

            for line in f:
                parts = line.strip().split(',')
                try:
                    vid_name = parts[0]
                    label    = int(float(parts[1]))
                    raw      = [float(x) for x in parts[3:]]
                except:
                    continue

                # Burun merkezleme
                nx = raw[idx_nose_x] if idx_nose_x >= 0 and idx_nose_x < len(raw) else 0.0
                ny = raw[idx_nose_y] if idx_nose_y >= 0 and idx_nose_y < len(raw) else 0.0

                norm_feat = []
                for i in valid_indices:
                    val = raw[i] if i < len(raw) else 0.0
                    if feature_names[i].endswith('_x'): val -= nx
                    elif feature_names[i].endswith('_y'): val -= ny
                    norm_feat.append(val)

                # Sabit normalizasyon (koşullu değil — ölçek tutarlılığı için)
                norm_feat = [v / 2.0 for v in norm_feat]

                # Boyut sabitle
                if len(norm_feat) > NEW_NUM_FEATURES:
                    norm_feat = norm_feat[:NEW_NUM_FEATURES]
                elif len(norm_feat) < NEW_NUM_FEATURES:
                    norm_feat.extend([0.0] * (NEW_NUM_FEATURES - len(norm_feat)))

                # Video değişti mi?
                if current_video is None:
                    current_video, current_label = vid_name, label

                if vid_name != current_video:
                    # Önceki videoyu yield et
                    if len(frames) > MAX_FRAMES:
                        frames = frames[:MAX_FRAMES]
                    else:
                        frames.extend([[0.0] * NEW_NUM_FEATURES] * (MAX_FRAMES - len(frames)))
                    yield (np.array(frames, dtype=np.float32),
                           tf.keras.utils.to_categorical(current_label, num_classes=NUM_CLASSES))
                    current_video, current_label, frames = vid_name, label, [norm_feat]
                else:
                    frames.append(norm_feat)

            # ✅ DÜZELTME: Son videoyu da yield et (önceden kayboluyordu)
            if current_video is not None and len(frames) > 0:
                if len(frames) > MAX_FRAMES:
                    frames = frames[:MAX_FRAMES]
                else:
                    frames.extend([[0.0] * NEW_NUM_FEATURES] * (MAX_FRAMES - len(frames)))
                yield (np.array(frames, dtype=np.float32),
                       tf.keras.utils.to_categorical(current_label, num_classes=NUM_CLASSES))

# ----------------- 6. DATASET -----------------
output_sig = (
    tf.TensorSpec(shape=(MAX_FRAMES, NEW_NUM_FEATURES), dtype=tf.float32),
    tf.TensorSpec(shape=(NUM_CLASSES,),                 dtype=tf.float32)
)

train_dataset = tf.data.Dataset.from_generator(
    lambda: robust_generator(TRAIN_CSV), output_signature=output_sig
).shuffle(buffer_size=500).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_generator(
    lambda: robust_generator(VAL_CSV), output_signature=output_sig
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# ----------------- 7. HIZLI BATCH KONTROLÜ -----------------
print("\n🧪 İlk batch kontrol ediliyor...")
for x, y in train_dataset.take(1):
    print(f"X shape: {x.shape} | Y shape: {y.shape}")
    print(f"X min/max: {x.numpy().min():.3f} / {x.numpy().max():.3f}")
    print(f"Label dağılımı (ilk 8): {tf.argmax(y, axis=1).numpy()[:8]}")
    nan_count = np.isnan(x.numpy()).sum()
    print(f"NaN sayısı: {nan_count} {'❌ SORUN VAR!' if nan_count > 0 else '✅'}")

# ----------------- 8. MODEL -----------------
CHECKPOINT_PATH = '/content/drive/MyDrive/isaret_dili_beyni3.h5'

if os.path.exists(CHECKPOINT_PATH):
    print(f"✅ Kayıtlı model bulundu, epoch 42'den devam ediliyor...")
    model = tf.keras.models.load_model(CHECKPOINT_PATH)
    # Optimizer'ı yeniden oluştur
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    print(f"📐 Learning rate: 0.0005 (epoch 42 seviyesi)")
else:
    print("⚠️ Kayıtlı model yok, sıfırdan başlanıyor...")
    model = Sequential([
        Input(shape=(MAX_FRAMES, NEW_NUM_FEATURES)),
        LSTM(128, return_sequences=True, activation='tanh'),
        BatchNormalization(),
        Dropout(0.3),
        LSTM(64, return_sequences=False, activation='tanh'),
        BatchNormalization(),
        Dropout(0.3),
        Dense(128, activation='relu'),
        Dropout(0.2),
        Dense(NUM_CLASSES, activation='softmax')
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

model.summary()

# ----------------- 9. EĞİTİM -----------------
checkpoint = ModelCheckpoint(
    '/content/drive/MyDrive/isaret_dili_beyni3.h5',
    monitor='val_accuracy', save_best_only=True, mode='max', verbose=1
)
early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
reduce_lr  = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1)

print(f"\n🚀 Epoch 42'den devam ediliyor!")

model.fit(
    train_dataset,
    steps_per_epoch=TRAIN_STEPS,
    validation_data=val_dataset,
    validation_steps=VAL_STEPS,
    epochs=100,
    initial_epoch=42,          # ← BURASI ÖNEMLİ, 43'ten başlar
    callbacks=[checkpoint, early_stop, reduce_lr]
)

🔄 Drive'a bağlanılıyor...
✅ Train: /content/drive/MyDrive/video_koordinat_verisetiTRAİN.csv
✅ Val:   /content/drive/MyDrive/video_koordinat_verisetival2.csv

🔍 CSV yapısı kontrol ediliyor...
Başlık (ilk 80 kar): video_name,class_id,frame_no,pose_0_x,pose_0_y,pose_0_z,pose_1_x,pose_1_y,pose_1
İlk satır (ilk 80 kar): signer0_sample1_color.mp4,41,1,0.5824792385101318,0.4149464964866638,-0.33358183
✅ CSV formatı doğru

📊 Video sayıları hesaplanıyor (biraz sürebilir)...
Train: 28132 video
Val:   4418 video

📐 Özellik sayısı: 201
📐 Train adım: 879 | Val adım: 138

🧪 İlk batch kontrol ediliyor...
X shape: (32, 100, 201) | Y shape: (32, 226)
X min/max: -0.627 / 0.265
Label dağılımı (ilk 8): [211   9 182  83 122 115 135 149]
NaN sayısı: 0 ✅


✅ Kayıtlı model bulundu, epoch 42'den devam ediliyor...
📐 Learning rate: 0.0005 (epoch 42 seviyesi)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 100, 128)       │       168,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 100, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 100, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 226)            │        29,154 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 256,610 (1002.38 KB)

 Trainable params: 256,226 (1000.88 KB)

... (3 outputs left out)


Epoch 44: finished saving model to /content/drive/MyDrive/isaret_dili_beyni3.h5
879/879 ━━━━━━━━━━━━━━━━━━━━ 444s 506ms/step - accuracy: 0.7976 - loss: 0.6299 - val_accuracy: 0.6730 - val_loss: 1.4692 - learning_rate: 5.0000e-04
Epoch 45/100
879/879 ━━━━━━━━━━━━━━━━━━━━ 0s 437ms/step - accuracy: 0.8182 - loss: 0.5574
Epoch 45: val_accuracy improved from 0.67301 to 0.67731, saving model to /content/drive/MyDrive/isaret_dili_beyni3.h5



Epoch 45: finished saving model to /content/drive/MyDrive/isaret_dili_beyni3.h5
879/879 ━━━━━━━━━━━━━━━━━━━━ 445s 507ms/step - accuracy: 0.8000 - loss: 0.6213 - val_accuracy: 0.6773 - val_loss: 1.5045 - learning_rate: 5.0000e-04
Epoch 46/100
879/879 ━━━━━━━━━━━━━━━━━━━━ 0s 436ms/step - accuracy: 0.8251 - loss: 0.5285
Epoch 46: val_accuracy improved from 0.67731 to 0.68274, saving model to /content/drive/MyDrive/isaret_dili_beyni3.h5



Epoch 46: finished saving model to /content/drive/MyDrive/isaret_dili_beyni3.h5
879/879 ━━━━━━━━━━━━━━━━━━━━ 451s 513ms/step - accuracy: 0.8044 - loss: 0.5975 - val_accuracy: 0.6827 - val_loss: 1.4481 - learning_rate: 5.0000e-04
Epoch 47/100
879/879 ━━━━━━━━━━━━━━━━━━━━ 0s 427ms/step - accuracy: 0.8326 - loss: 0.5109
Epoch 47: val_accuracy did not improve from 0.68274
879/879 ━━━━━━━━━━━━━━━━━━━━ 446s 508ms/step - accuracy: 0.8131 - loss: 0.5712 - val_accuracy: 0.6703 - val_loss: 1.5584 - learning_rate: 5.0000e-04
Epoch 48/100
879/879 ━━━━━━━━━━━━━━━━━━━━ 0s 423ms/step - accuracy: 0.8305 - loss: 0.5182
Epoch 48: val_accuracy did not improve from 0.68274
879/879 ━━━━━━━━━━━━━━━━━━━━ 462s 526ms/step - accuracy: 0.8129 - loss: 0.5790 - val_accuracy: 0.6796 - val_loss: 1.4812 - learning_rate: 5.0000e-04
Epoch 49/100
 92/879 ━━━━━━━━━━━━━━━━━━━━ 2:10 166ms/step - accuracy: 0.8368 - loss: 0.5014

KeyboardInterrupt: 

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
from google.colab import drive

# ----------------- 1. BAĞLANTI -----------------
print("🔄 Drive'a bağlanılıyor...")
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

TRAIN_CSV = '/content/drive/MyDrive/video_koordinat_verisetiTRAİN.csv'
VAL_CSV   = '/content/drive/MyDrive/video_koordinat_verisetival2.csv'
TEST_CSV  = '/content/drive/MyDrive/video_koordinat_veriseti_TEST.csv'

for p in [TRAIN_CSV, VAL_CSV, TEST_CSV]:
    assert os.path.exists(p), f"❌ Bulunamadı: {p}"
    print(f"✅ {p}")

# ----------------- 2. DOĞRULAMA -----------------
with open(TRAIN_CSV, 'r', encoding='utf-8') as f:
    header_line = f.readline().strip()
    first_line  = f.readline().strip()

print(f"\nBaşlık (ilk 80 kar): {header_line[:80]}")
print(f"İlk satır (ilk 80 kar): {first_line[:80]}")
assert 'video_name' in header_line and 'class_id' in header_line
print("✅ CSV formatı doğru")

# ----------------- 3. VİDEO SAYISI -----------------
def count_videos(filepath):
    count, prev = 0, None
    with open(filepath, 'r', encoding='utf-8') as f:
        f.readline()
        for line in f:
            parts = line.strip().split(',')
            if len(parts) > 0 and parts[0] != prev:
                count += 1
                prev = parts[0]
    return count

print("\n📊 Video sayıları hesaplanıyor...")
train_count = count_videos(TRAIN_CSV)
val_count   = count_videos(VAL_CSV)
test_count  = count_videos(TEST_CSV)
total_count = train_count + val_count + test_count
print(f"Train: {train_count} | Val: {val_count} | Test: {test_count} | Toplam: {total_count}")

# ----------------- 4. AYARLAR -----------------
with open(TRAIN_CSV, 'r', encoding='utf-8') as f:
    header = f.readline().strip().split(',')

feature_names = header[3:]

bad_indices = set()
for i, col_name in enumerate(feature_names):
    if 'pose_' in col_name:
        try:
            if int(col_name.split('_')[1]) >= 25:
                bad_indices.add(i)
        except: pass

valid_indices    = [i for i in range(len(feature_names)) if i not in bad_indices]
NEW_NUM_FEATURES = len(valid_indices)

idx_nose_x = feature_names.index('pose_0_x') if 'pose_0_x' in feature_names else -1
idx_nose_y = feature_names.index('pose_0_y') if 'pose_0_y' in feature_names else -1

NUM_CLASSES  = 226
MAX_FRAMES   = 100
BATCH_SIZE   = 32
TOTAL_STEPS  = total_count // BATCH_SIZE

print(f"\n📐 Özellik sayısı: {NEW_NUM_FEATURES}")
print(f"📐 Toplam adım/epoch: {TOTAL_STEPS}")

# ----------------- 5. GENERATOR (3 dosya birleşik) -----------------
def combined_generator(fp1, fp2, fp3):
    while True:
        for filepath in [fp1, fp2, fp3]:
            with open(filepath, 'r', encoding='utf-8', errors='replace') as f:
                f.readline()
                current_video, current_label, frames = None, None, []

                for line in f:
                    parts = line.strip().split(',')
                    try:
                        vid_name = parts[0]
                        label    = int(float(parts[1]))
                        raw      = [float(x) for x in parts[3:]]
                    except:
                        continue

                    nx = raw[idx_nose_x] if idx_nose_x >= 0 and idx_nose_x < len(raw) else 0.0
                    ny = raw[idx_nose_y] if idx_nose_y >= 0 and idx_nose_y < len(raw) else 0.0

                    norm_feat = []
                    for i in valid_indices:
                        val = raw[i] if i < len(raw) else 0.0
                        if feature_names[i].endswith('_x'): val -= nx
                        elif feature_names[i].endswith('_y'): val -= ny
                        norm_feat.append(val / 2.0)

                    if len(norm_feat) > NEW_NUM_FEATURES:
                        norm_feat = norm_feat[:NEW_NUM_FEATURES]
                    elif len(norm_feat) < NEW_NUM_FEATURES:
                        norm_feat.extend([0.0] * (NEW_NUM_FEATURES - len(norm_feat)))

                    if current_video is None:
                        current_video, current_label = vid_name, label

                    if vid_name != current_video:
                        if len(frames) > MAX_FRAMES:
                            frames = frames[:MAX_FRAMES]
                        else:
                            frames.extend([[0.0] * NEW_NUM_FEATURES] * (MAX_FRAMES - len(frames)))
                        yield (np.array(frames, dtype=np.float32),
                               tf.keras.utils.to_categorical(current_label, num_classes=NUM_CLASSES))
                        current_video, current_label, frames = vid_name, label, [norm_feat]
                    else:
                        frames.append(norm_feat)

                if current_video is not None and len(frames) > 0:
                    if len(frames) > MAX_FRAMES:
                        frames = frames[:MAX_FRAMES]
                    else:
                        frames.extend([[0.0] * NEW_NUM_FEATURES] * (MAX_FRAMES - len(frames)))
                    yield (np.array(frames, dtype=np.float32),
                           tf.keras.utils.to_categorical(current_label, num_classes=NUM_CLASSES))

# ----------------- 6. DATASET -----------------
output_sig = (
    tf.TensorSpec(shape=(MAX_FRAMES, NEW_NUM_FEATURES), dtype=tf.float32),
    tf.TensorSpec(shape=(NUM_CLASSES,),                 dtype=tf.float32)
)

combined_dataset = tf.data.Dataset.from_generator(
    lambda: combined_generator(TRAIN_CSV, VAL_CSV, TEST_CSV),
    output_signature=output_sig
).shuffle(buffer_size=500).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# ----------------- 7. BATCH KONTROLÜ -----------------
print("\n🧪 İlk batch kontrol ediliyor...")
for x, y in combined_dataset.take(1):
    print(f"X shape: {x.shape} | Y shape: {y.shape}")
    print(f"X min/max: {x.numpy().min():.3f} / {x.numpy().max():.3f}")
    print(f"Label dağılımı (ilk 8): {tf.argmax(y, axis=1).numpy()[:8]}")
    nan_count = np.isnan(x.numpy()).sum()
    print(f"NaN sayısı: {nan_count} {'❌ SORUN VAR!' if nan_count > 0 else '✅'}")

# ----------------- 8. MODEL -----------------
# Beyni3'ü yükle, kaldığı yerden (epoch 42) devam et
CHECKPOINT_PATH = '/content/drive/MyDrive/isaret_dili_beyni3.h5'

assert os.path.exists(CHECKPOINT_PATH), "❌ beyni3.h5 bulunamadı!"
print(f"\n✅ beyni3.h5 yükleniyor (epoch 42, val_acc ~%67)...")
model = tf.keras.models.load_model(CHECKPOINT_PATH)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),  # epoch 42 seviyesi
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print("📐 Learning rate: 0.0005 (epoch 42'den devam)")
model.summary()

# ----------------- 9. EĞİTİM -----------------
checkpoint = ModelCheckpoint(
    '/content/drive/MyDrive/isaret_dili_beyni_FINAL.h5',
    monitor='loss',
    save_best_only=True,
    mode='min',
    verbose=1
)
early_stop = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
reduce_lr  = ReduceLROnPlateau(monitor='loss', factor=0.5, patience=5, verbose=1)

print(f"\n🚀 Epoch 42'den devam, tüm veriyle FINAL eğitim!")
print(f"   Toplam: {total_count} video → {TOTAL_STEPS} adım/epoch")

model.fit(
    combined_dataset,
    steps_per_epoch=TOTAL_STEPS,
    epochs=150,
    initial_epoch=42,   # beyni3 epoch 42'de bırakmıştı
    callbacks=[checkpoint, early_stop, reduce_lr]
)

🔄 Drive'a bağlanılıyor...
✅ /content/drive/MyDrive/video_koordinat_verisetiTRAİN.csv
✅ /content/drive/MyDrive/video_koordinat_verisetival2.csv
✅ /content/drive/MyDrive/video_koordinat_veriseti_TEST.csv

Başlık (ilk 80 kar): video_name,class_id,frame_no,pose_0_x,pose_0_y,pose_0_z,pose_1_x,pose_1_y,pose_1
İlk satır (ilk 80 kar): signer0_sample1_color.mp4,41,1,0.5824792385101318,0.4149464964866638,-0.33358183
✅ CSV formatı doğru

📊 Video sayıları hesaplanıyor...
Train: 28132 | Val: 4418 | Test: 2808 | Toplam: 35358

📐 Özellik sayısı: 201
📐 Toplam adım/epoch: 1104

🧪 İlk batch kontrol ediliyor...
X shape: (32, 100, 201) | Y shape: (32, 226)
X min/max: -0.643 / 0.269
Label dağılımı (ilk 8): [ 15  68 100 116   2  38  28 189]
NaN sayısı: 0 ✅



✅ beyni3.h5 yükleniyor (epoch 42, val_acc ~%67)...
📐 Learning rate: 0.0005 (epoch 42'den devam)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 100, 128)       │       168,960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 100, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 100, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 226)            │        29,154 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 256,610 (1002.38 KB)

 Trainable params: 256,226 (1000.88 KB)

... (6 outputs left out)


Epoch 47: finished saving model to /content/drive/MyDrive/isaret_dili_beyni_FINAL.h5
1104/1104 ━━━━━━━━━━━━━━━━━━━━ 463s 419ms/step - accuracy: 0.7959 - loss: 0.6688 - learning_rate: 5.0000e-04
Epoch 48/150
1104/1104 ━━━━━━━━━━━━━━━━━━━━ 0s 416ms/step - accuracy: 0.8175 - loss: 0.5699
Epoch 48: loss improved from 0.66880 to 0.64222, saving model to /content/drive/MyDrive/isaret_dili_beyni_FINAL.h5



Epoch 48: finished saving model to /content/drive/MyDrive/isaret_dili_beyni_FINAL.h5
1104/1104 ━━━━━━━━━━━━━━━━━━━━ 461s 418ms/step - accuracy: 0.7995 - loss: 0.6422 - learning_rate: 5.0000e-04
Epoch 49/150
1104/1104 ━━━━━━━━━━━━━━━━━━━━ 0s 421ms/step - accuracy: 0.8285 - loss: 0.5444
Epoch 49: loss improved from 0.64222 to 0.62083, saving model to /content/drive/MyDrive/isaret_dili_beyni_FINAL.h5



Epoch 49: finished saving model to /content/drive/MyDrive/isaret_dili_beyni_FINAL.h5
1104/1104 ━━━━━━━━━━━━━━━━━━━━ 466s 423ms/step - accuracy: 0.8090 - loss: 0.6208 - learning_rate: 5.0000e-04
Epoch 50/150
1104/1104 ━━━━━━━━━━━━━━━━━━━━ 0s 417ms/step - accuracy: 0.8290 - loss: 0.5383
Epoch 50: loss improved from 0.62083 to 0.61530, saving model to /content/drive/MyDrive/isaret_dili_beyni_FINAL.h5



Epoch 50: finished saving model to /content/drive/MyDrive/isaret_dili_beyni_FINAL.h5
1104/1104 ━━━━━━━━━━━━━━━━━━━━ 462s 419ms/step - accuracy: 0.8082 - loss: 0.6153 - learning_rate: 5.0000e-04
Epoch 51/150
 854/1104 ━━━━━━━━━━━━━━━━━━━━ 1:44 417ms/step - accuracy: 0.8377 - loss: 0.5116